In [1]:
import pandas as pd
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error
)
import numpy as np
import re
import ast

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

path = "../outputs/lactate_extraction_results.csv"

In [27]:
df = pd.read_csv(path)
#df = df.loc[df['n_chunks'] >= 1, :]

discharge = pd.read_csv("../../Outputs/discharge_filtered.csv")
df_final_structured = pd.read_csv("../../Outputs/final_merged_dataset_with_missing.csv")
df_full = discharge.merge(df_final_structured, how="inner", on=["hadm_id", "subject_id"])


discharge = pd.read_csv("../../Outputs/discharge_filtered.csv")
df_final_structured_not_missing = pd.read_csv("../../Outputs/final_structured_dataset.csv")
df_full_not_missing = discharge.merge(df_final_structured_not_missing, how="inner", on=["hadm_id", "subject_id"])

In [3]:
#df.head()

## Alignment between Structured and LLM Lactate

In [4]:
eval_df = df.copy()

eval_df = eval_df[eval_df["llm_present"].notna() & eval_df["structured_lactate"].notna()].copy()

eval_df["pred_present"] = eval_df["llm_present"].astype(bool)
eval_df["gold_present"] = eval_df["structured_lactate"] > 2.0

print("N =", len(eval_df))
print("Accuracy:", accuracy_score(eval_df["gold_present"], eval_df["pred_present"]))
print("Precision:", precision_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))
print("Recall:", recall_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))
print("F1:", f1_score(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(eval_df["gold_present"], eval_df["pred_present"]))

print("\nClassification report:")
print(classification_report(eval_df["gold_present"], eval_df["pred_present"], zero_division=0))

N = 927
Accuracy: 0.7281553398058253
Precision: 0.7693430656934307
Recall: 0.8486312399355878
F1: 0.8070444104134763

Confusion matrix:
[[148 158]
 [ 94 527]]

Classification report:
              precision    recall  f1-score   support

       False       0.61      0.48      0.54       306
        True       0.77      0.85      0.81       621

    accuracy                           0.73       927
   macro avg       0.69      0.67      0.67       927
weighted avg       0.72      0.73      0.72       927



In [5]:
value_df = df.copy()
value_df = value_df[
    value_df["llm_lactate_value"].notna() &
    value_df["structured_lactate"].notna()
].copy()

print("N with comparable values =", len(value_df))
print("MAE:", mean_absolute_error(value_df["structured_lactate"], value_df["llm_lactate_value"]))
print("RMSE:", np.sqrt(mean_squared_error(value_df["structured_lactate"], value_df["llm_lactate_value"])))
print("Correlation:", value_df["structured_lactate"].corr(value_df["llm_lactate_value"]))

value_df["abs_error"] = (value_df["structured_lactate"] - value_df["llm_lactate_value"]).abs()
value_df["signed_error"] = value_df["llm_lactate_value"] - value_df["structured_lactate"]

N with comparable values = 957
MAE: 2.849903294439171
RMSE: 4.114895210325117
Correlation: 0.5761167298356504


# How well Lactate was extracted from the notes?

In [6]:
def normalize_chunks(chunks):
    if pd.isna(chunks):
        return ""
    if isinstance(chunks, str):
        try:
            parsed = ast.literal_eval(chunks)
            if isinstance(parsed, list):
                return " ".join(str(x) for x in parsed)
        except:
            pass
        return chunks
    if isinstance(chunks, list):
        return " ".join(str(x) for x in chunks)
    return str(chunks)

def extract_lactate_values(text):
    text = str(text).lower()

    pattern = r"(?:lactate|lactic acid)\s*(?:was|is|of|:|=|-)?\s*([0-9]+(?:\.[0-9]+)?)"

    values = []
    for m in re.findall(pattern, text):
        try:
            values.append(float(m))
        except:
            pass

    return values

def derive_chunk_gold_label(text):
    t = str(text).lower().strip()
    if not t:
        return None
    
    if not re.search(r"\b(lactate|lactic acid|lactic acidosis)\b", t):
        return None

    if re.search(r"lactic acidosis|elevated lactate|high lactate|lactate (?:is |was )?(?:elevated|high)", t):
        return True

    if re.search(r"normal lactate|lactate (?:is |was )?(?:normal|cleared|normalized|not elevated)|lactate cleared|lactate normalized", t):
        return False

    values = extract_lactate_values(t)
    if values:
        if any(v > 2.0 for v in values):
            return True
        if all(v <= 2.0 for v in values):
            return False

    return None

def normalize_llm_present(x):
    if pd.isna(x):
        return None
    if x is True or x is False:
        return x
    if isinstance(x, str):
        x = x.strip().lower()
        if x == "true":
            return True
        if x == "false":
            return False
    return None

df["chunk_text"] = df["chunks"].apply(normalize_chunks)
df["gold_present_from_chunks"] = df["chunk_text"].apply(derive_chunk_gold_label)
df["llm_present_norm"] = df["llm_present"].apply(normalize_llm_present)

In [7]:
# Mention detection
df["gold_mentioned"] = df["chunk_text"].str.lower().str.contains(
    r"\b(?:lactate|lactic acid|lactic acidosis)\b",
    regex=True,
    na=False
)
df["llm_mentioned"] = df["llm_present_norm"].notna()

print("=== Mention detection ===")
print(classification_report(df["gold_mentioned"], df["llm_mentioned"], zero_division=0))

=== Mention detection ===
              precision    recall  f1-score   support

       False       0.83      1.00      0.91       577
        True       1.00      0.89      0.94      1041

    accuracy                           0.93      1618
   macro avg       0.91      0.94      0.92      1618
weighted avg       0.94      0.93      0.93      1618



In [8]:
print("=== Elevated detection ===")
compare_df = df[
    df["gold_present_from_chunks"].notna() &
    df["llm_present_norm"].notna()
].copy()

y_true = compare_df["gold_present_from_chunks"].astype(bool)
y_pred = compare_df["llm_present_norm"].astype(bool)

print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

=== Elevated detection ===
              precision    recall  f1-score   support

       False       0.62      0.94      0.75       158
        True       0.99      0.88      0.93       753

    accuracy                           0.89       911
   macro avg       0.81      0.91      0.84       911
weighted avg       0.92      0.89      0.90       911

[[149   9]
 [ 90 663]]


In [9]:
mismatch_df = compare_df[
    compare_df["gold_present_from_chunks"] != compare_df["llm_present_norm"]
]

mismatchs = mismatch_df.head(20)[["chunk_text","gold_present_from_chunks","llm_present_norm","llm_evidence_quote"]]

In [10]:
def return_text_llm_answer_gold_answer(df):
    for idx, row in df.iterrows():
        print(row['chunk_text'])
        print('gold: ', row['gold_present_from_chunks'])
        print('llm_present: ', row['llm_present_norm'])
        print('llm_evidence_quote: ', row['llm_evidence_quote'])
        print('*******************************************************')

In [11]:
#return_text_llm_answer_gold_answer(mismatchs)

In [12]:
df["llm_elevated_from_value"] = df["llm_lactate_value"] > 2

compare_df = df[
    df["gold_present_from_chunks"].notna() &
    df["llm_lactate_value"].notna()
]

y_true = compare_df["gold_present_from_chunks"].astype(bool)
y_pred = compare_df["llm_elevated_from_value"].astype(bool)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

       False       0.90      0.95      0.92       169
        True       0.99      0.98      0.98       775

    accuracy                           0.97       944
   macro avg       0.95      0.96      0.95       944
weighted avg       0.97      0.97      0.97       944



In [13]:
mismatch_df = compare_df[
    compare_df["gold_present_from_chunks"] != compare_df["llm_elevated_from_value"]
]

mismatchs = mismatch_df.head(20)[["chunk_text","gold_present_from_chunks","llm_elevated_from_value","llm_evidence_quote"]]

In [14]:
#mismatchs

In [15]:
#for idx, row in mismatchs.iterrows():
#    print(row['chunk_text'])
#    print('gold: ', row['gold_present_from_chunks'])
#    print('llm_present: ', row['llm_elevated_from_value'])
#    print('llm_evidence_quote: ', row['llm_evidence_quote'])
#    print('*******************************************************')

## Added Value for Shock

In [28]:
new_path = '../outputs/shock_extraction_results_2026-03-19_13-24-05.csv'
df = pd.read_csv(new_path)
df_added_value = df.merge(df_full_not_missing, on='note_id', how='inner')

In [29]:
#df_added_value.to_csv('df_added_value_shock.csv')

### strict structured shock proxy, treat llm_present as binary

In [31]:
# -----------------------------
# 1. Load data
# -----------------------------
df = df_added_value.copy()

# -----------------------------
# 2. Normalize LLM columns
# -----------------------------
def norm_present(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"true", "1", "yes"}:
        return 1
    if s in {"false", "0", "no"}:
        return 0
    return np.nan

sev_map = {
    "none": 0,
    "mild": 1,
    "moderate": 2,
    "severe": 3
}

df["llm_present_bin"] = df["llm_present"].map(norm_present)

df["llm_severity_norm"] = (
    df["llm_severity"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(sev_map)
)

# -----------------------------
# 3. Normalize outcome
# -----------------------------
def norm_outcome(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"dead", "deceased", "1", "true", "yes"}:
        return 1
    if s in {"alive", "0", "false", "no"}:
        return 0
    try:
        return int(float(s))
    except:
        return np.nan

df["outcome"] = df["survival_status"].map(norm_outcome)

# -----------------------------
# 4. Variable groups
# -----------------------------
continuous_vars = [
    "anchor_age",
    "SBP",
    "MBP",
    "Lactate",
    "pH",
    "Bicarbonate",
    "AnionGap",
]

binary_structured_vars = [
    "received_epinephrine",
    "received_dopamine",
]

llm_binary_vars = ["llm_present_bin"]
llm_ordinal_vars = ["llm_severity_norm"]

structured_vars = continuous_vars + binary_structured_vars
model_b_vars = structured_vars + llm_binary_vars
model_c_vars = structured_vars + llm_binary_vars + llm_ordinal_vars

# -----------------------------
# 5. Descriptive analysis
# -----------------------------
print("Outcome counts:")
print(df["outcome"].value_counts(dropna=False))
print("**********************************************************************")

print("LLM present counts:")
print(df["llm_present_bin"].value_counts(dropna=False))
print("**********************************************************************")

print("LLM severity counts:")
print(df["llm_severity_norm"].value_counts(dropna=False).sort_index())
print("**********************************************************************")

print("Mortality by llm_present:")
print(df.groupby("llm_present_bin")["outcome"].agg(["count", "mean"]))
print("**********************************************************************")

# -----------------------------
# 6. Complete-case datasets
# -----------------------------
cc_struct = df[["outcome"] + structured_vars].dropna().copy()
cc_model_b = df[["outcome"] + model_b_vars].dropna().copy()
cc_model_c = df[["outcome"] + model_c_vars].dropna().copy()

print("Complete-case dataset sizes:")
print("Structured only:", cc_struct.shape)
print("Structured + llm_present:", cc_model_b.shape)
print("Structured + llm_present + severity:", cc_model_c.shape)
print("**********************************************************************")

# -----------------------------
# 7. CV AUROC with scaling only continuous variables
# -----------------------------
def cv_auc(data, features, continuous_features):
    X = data[features]
    y = data["outcome"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), continuous_features)
        ],
        remainder="passthrough"   # binary/ordinal features unchanged
    )

    model = Pipeline([
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=5000))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return aucs.mean(), aucs.std()

# Structured only
auc_struct, sd_struct = cv_auc(cc_struct, structured_vars, continuous_vars)
print(f"Structured only (all complete cases): AUROC = {auc_struct:.3f} ± {sd_struct:.3f}")
print("**********************************************************************")

# Fair same-row comparison for llm_present subset
auc_struct_b, sd_struct_b = cv_auc(cc_model_b, structured_vars, continuous_vars)
auc_plus_llm_b, sd_plus_llm_b = cv_auc(cc_model_b, model_b_vars, continuous_vars)

print("\nSame-row comparison: structured vs structured + llm_present")
print(f"Structured only:            AUROC = {auc_struct_b:.3f} ± {sd_struct_b:.3f}")
print(f"Structured + llm_present:   AUROC = {auc_plus_llm_b:.3f} ± {sd_plus_llm_b:.3f}")
print("**********************************************************************")

# Fair same-row comparison for llm_present + severity subset
auc_struct_c, sd_struct_c = cv_auc(cc_model_c, structured_vars, continuous_vars)
auc_plus_llm_c, sd_plus_llm_c = cv_auc(cc_model_c, model_c_vars, continuous_vars)

print("\nSame-row comparison: structured vs structured + llm_present + severity")
print(f"Structured only:                        AUROC = {auc_struct_c:.3f} ± {sd_struct_c:.3f}")
print(f"Structured + llm_present + severity:    AUROC = {auc_plus_llm_c:.3f} ± {sd_plus_llm_c:.3f}")
print("**********************************************************************")

# -----------------------------
# 8. Statsmodels logistic regression
#    (no scaling needed here for inference)
# -----------------------------
def fit_logit(data, features):
    X = data[features].copy()
    X = sm.add_constant(X, has_constant="add")
    y = data["outcome"]
    model = sm.Logit(y, X).fit(disp=False)
    return model

print("\nMultivariable model: structured + llm_present")
model_b = fit_logit(cc_model_b, model_b_vars)
print(model_b.summary2().tables[1].loc[["llm_present_bin"]])
print("**********************************************************************")

print("\nMultivariable model: structured + llm_present + severity")
model_c = fit_logit(cc_model_c, model_c_vars)
print(model_c.summary2().tables[1].loc[["llm_present_bin", "llm_severity_norm"]])
print("**********************************************************************")

# -----------------------------
# 9. Strict structured shock proxy
# -----------------------------
proxy_vars = [
    "SBP",
    "MBP",
    "Lactate",
    "pH",
    "Bicarbonate",
    "AnionGap",
    "received_epinephrine",
    "received_dopamine"
]

# only classify patients if ALL proxy vars are observed
complete_proxy = df[proxy_vars].notna().all(axis=1)

proxy_positive = (
    (df["SBP"] < 90) |
    (df["MBP"] < 65) |
    (df["Lactate"] > 2) |
    (df["pH"] < 7.35) |
    (df["Bicarbonate"] < 22) |
    (df["AnionGap"] > 16) |
    (df["received_epinephrine"] == 1) |
    (df["received_dopamine"] == 1)
)

df["structured_shock_proxy_strict"] = np.where(
    complete_proxy,
    np.where(proxy_positive, 1, 0),
    np.nan
)

print("Strict structured shock proxy counts:")
print(df["structured_shock_proxy_strict"].value_counts(dropna=False))
print("**********************************************************************")

sub_strict = df[df["structured_shock_proxy_strict"] == 0].copy()

print("\nPatients not flagged by STRICT structured shock proxy:")
print(sub_strict["llm_present_bin"].value_counts(dropna=False))
print("**********************************************************************")

print("Mortality by llm_present within STRICT structured-proxy-negative subgroup:")
print(sub_strict.groupby("llm_present_bin")["outcome"].agg(["count", "mean"]))

Outcome counts:
outcome
0    815
1    803
Name: count, dtype: int64
**********************************************************************
LLM present counts:
llm_present_bin
NaN    986
1.0    518
0.0    114
Name: count, dtype: int64
**********************************************************************
LLM severity counts:
llm_severity_norm
0.0      35
1.0       1
2.0      15
3.0     497
NaN    1070
Name: count, dtype: int64
**********************************************************************
Mortality by llm_present:
                 count      mean
llm_present_bin                 
0.0                114  0.298246
1.0                518  0.575290
**********************************************************************
Complete-case dataset sizes:
Structured only: (1618, 10)
Structured + llm_present: (632, 11)
Structured + llm_present + severity: (548, 12)
**********************************************************************
Structured only (all complete cases): AUROC = 0.709 ± 0.042

### treat missing values as a new category, impute missing values

In [32]:
# -----------------------------
# 1. Load data
# -----------------------------
df = df_added_value.copy()

# -----------------------------
# 2. Normalize LLM columns
# -----------------------------
def norm_present_3cat(x):
    """
    1 = present
    0 = absent
    -1 = missing / no usable LLM output
    """
    if pd.isna(x):
        return "-1"
    s = str(x).strip().lower()
    if s in {"true", "1", "yes"}:
        return "1"
    if s in {"false", "0", "no"}:
        return "0"
    return "-1"

sev_valid = {"none", "mild", "moderate", "severe"}

df["llm_present_3cat"] = df["llm_present"].map(norm_present_3cat)

df["llm_severity_cat"] = (
    df["llm_severity"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df.loc[~df["llm_severity_cat"].isin(sev_valid), "llm_severity_cat"] = "missing"

# Optional binary version for descriptive comparison if needed later
def norm_present_binary(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"true", "1", "yes"}:
        return 1
    if s in {"false", "0", "no"}:
        return 0
    return np.nan

df["llm_present_bin"] = df["llm_present"].map(norm_present_binary)

# -----------------------------
# 3. Normalize outcome
# -----------------------------
def norm_outcome(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"dead", "deceased", "1", "true", "yes"}:
        return 1
    if s in {"alive", "0", "false", "no"}:
        return 0
    try:
        return int(float(s))
    except:
        return np.nan

df["outcome"] = df["survival_status"].map(norm_outcome)

# Keep only rows with known outcome
df = df[df["outcome"].notna()].copy()

# -----------------------------
# 4. Variable groups
# -----------------------------
continuous_vars = [
    "anchor_age",
    "SBP",
    "MBP",
    "Lactate",
    "pH",
    "Bicarbonate",
    "AnionGap",
]

binary_structured_vars = [
    "received_epinephrine",
    "received_dopamine",
]

categorical_llm_present = ["llm_present_3cat"]      # "-1", "0", "1"
categorical_llm_severity = ["llm_severity_cat"]     # "missing", "none", "mild", "moderate", "severe"

structured_vars = continuous_vars + binary_structured_vars
model_b_vars = structured_vars + categorical_llm_present
model_c_vars = structured_vars + categorical_llm_present + categorical_llm_severity

# -----------------------------
# 5. Descriptive analysis
# -----------------------------
print("Outcome counts:")
print(df["outcome"].value_counts(dropna=False))
print("**********************************************************************")

print("LLM present 3-category counts:")
print(df["llm_present_3cat"].value_counts(dropna=False).sort_index())
print("**********************************************************************")

print("LLM severity category counts:")
print(df["llm_severity_cat"].value_counts(dropna=False))
print("**********************************************************************")

print("Mortality by llm_present_3cat:")
print(df.groupby("llm_present_3cat")["outcome"].agg(["count", "mean"]))
print("**********************************************************************")

# -----------------------------
# 6. Preprocessing pipelines
# -----------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

binary_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

llm_present_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        categories=[["-1", "0", "1"]],
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    ))
])

llm_severity_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        categories=[["missing", "none", "mild", "moderate", "severe"]],
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    ))
])

def make_preprocessor(use_llm_present=False, use_llm_severity=False):
    transformers = [
        ("num", numeric_transformer, continuous_vars),
        ("bin", binary_transformer, binary_structured_vars),
    ]

    if use_llm_present:
        transformers.append(("llm_present_cat", llm_present_transformer, categorical_llm_present))

    if use_llm_severity:
        transformers.append(("llm_severity_cat", llm_severity_transformer, categorical_llm_severity))

    return ColumnTransformer(transformers=transformers, remainder="drop")

# -----------------------------
# 7. Cross-validated AUROC
# -----------------------------
def cv_auc(data, use_llm_present=False, use_llm_severity=False):
    if use_llm_present and use_llm_severity:
        X = data[model_c_vars]
    elif use_llm_present:
        X = data[model_b_vars]
    else:
        X = data[structured_vars]

    y = data["outcome"]

    preprocessor = make_preprocessor(
        use_llm_present=use_llm_present,
        use_llm_severity=use_llm_severity
    )

    model = Pipeline([
        ("preprocess", preprocessor),
        ("clf", LogisticRegression(max_iter=5000))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return aucs.mean(), aucs.std()

auc_struct, sd_struct = cv_auc(df, use_llm_present=False, use_llm_severity=False)
auc_llm_present, sd_llm_present = cv_auc(df, use_llm_present=True, use_llm_severity=False)
auc_llm_both, sd_llm_both = cv_auc(df, use_llm_present=True, use_llm_severity=True)

print("AUROC comparison with imputation + missing-aware LLM variables")
print(f"Structured only:                          {auc_struct:.3f} ± {sd_struct:.3f}")
print(f"Structured + llm_present_3cat:           {auc_llm_present:.3f} ± {sd_llm_present:.3f}")
print(f"Structured + llm_present_3cat + severity:{auc_llm_both:.3f} ± {sd_llm_both:.3f}")
print("**********************************************************************")

# -----------------------------
# 8. Statsmodels inference
# -----------------------------
df_sm = df.copy()

# Continuous: coerce numeric + median impute
for col in continuous_vars:
    df_sm[col] = pd.to_numeric(df_sm[col], errors="coerce")
    df_sm[col] = df_sm[col].fillna(df_sm[col].median())

# Binary structured: coerce numeric + mode impute
for col in binary_structured_vars:
    df_sm[col] = pd.to_numeric(df_sm[col], errors="coerce")
    mode_val = df_sm[col].mode(dropna=True)
    if len(mode_val) > 0:
        df_sm[col] = df_sm[col].fillna(mode_val.iloc[0])

# Ensure categorical fields are strings
df_sm["llm_present_3cat"] = df_sm["llm_present_3cat"].astype(str)
df_sm["llm_severity_cat"] = df_sm["llm_severity_cat"].astype(str)

# One-hot encode LLM present
present_dummies = pd.get_dummies(
    df_sm["llm_present_3cat"],
    prefix="llm_present",
    drop_first=True,
    dtype=int
)

# One-hot encode severity
severity_dummies = pd.get_dummies(
    df_sm["llm_severity_cat"],
    prefix="llm_severity",
    drop_first=True,
    dtype=int
)

# -------- Model B: structured + llm_present_3cat --------
X_b = pd.concat(
    [df_sm[structured_vars], present_dummies],
    axis=1
)

X_b = X_b.apply(pd.to_numeric, errors="coerce").astype(float)

tmp_b = pd.concat([df_sm["outcome"], X_b], axis=1).dropna()
y_b = tmp_b["outcome"].astype(int)
X_b = tmp_b.drop(columns=["outcome"])

X_b = sm.add_constant(X_b, has_constant="add")
model_b = sm.Logit(y_b, X_b).fit(disp=False)

print("\nStatsmodels: structured + llm_present_3cat")
print(model_b.summary2().tables[1][["Coef.", "Std.Err.", "z", "P>|z|"]])
print("**********************************************************************")

# -------- Model C: structured + llm_present_3cat + severity --------
X_c = pd.concat(
    [df_sm[structured_vars], present_dummies, severity_dummies],
    axis=1
)

X_c = X_c.apply(pd.to_numeric, errors="coerce").astype(float)

tmp_c = pd.concat([df_sm["outcome"], X_c], axis=1).dropna()
y_c = tmp_c["outcome"].astype(int)
X_c = tmp_c.drop(columns=["outcome"])

X_c = sm.add_constant(X_c, has_constant="add")
model_c = sm.Logit(y_c, X_c).fit(disp=False)

print("\nStatsmodels: structured + llm_present_3cat + llm_severity_cat")
print(model_c.summary2().tables[1][["Coef.", "Std.Err.", "z", "P>|z|"]])
print("**********************************************************************")

# -----------------------------
# 9. Strict structured shock proxy
# -----------------------------
proxy_vars = [
    "SBP",
    "MBP",
    "Lactate",
    "pH",
    "Bicarbonate",
    "AnionGap",
    "received_epinephrine",
    "received_dopamine"
]

complete_proxy = df[proxy_vars].notna().all(axis=1)

proxy_positive = (
    (df["SBP"] < 90) |
    (df["MBP"] < 65) |
    (df["Lactate"] > 2) |
    (df["pH"] < 7.35) |
    (df["Bicarbonate"] < 22) |
    (df["AnionGap"] > 16) |
    (df["received_epinephrine"] == 1) |
    (df["received_dopamine"] == 1)
)

df["structured_shock_proxy_strict"] = np.where(
    complete_proxy,
    np.where(proxy_positive, 1, 0),
    np.nan
)

print("Strict structured shock proxy counts:")
print(df["structured_shock_proxy_strict"].value_counts(dropna=False))
print("**********************************************************************")

sub_strict = df[df["structured_shock_proxy_strict"] == 0].copy()

print("\nPatients not flagged by STRICT structured shock proxy:")
print(sub_strict["llm_present_3cat"].value_counts(dropna=False).sort_index())
print("**********************************************************************")

print("Mortality by llm_present_3cat within STRICT structured-proxy-negative subgroup:")
print(sub_strict.groupby("llm_present_3cat")["outcome"].agg(["count", "mean"]))

Outcome counts:
outcome
0    815
1    803
Name: count, dtype: int64
**********************************************************************
LLM present 3-category counts:
llm_present_3cat
-1    986
0     114
1     518
Name: count, dtype: int64
**********************************************************************
LLM severity category counts:
llm_severity_cat
missing     1070
severe       497
none          35
moderate      15
mild           1
Name: count, dtype: int64
**********************************************************************
Mortality by llm_present_3cat:
                  count      mean
llm_present_3cat                 
-1                  986  0.477688
0                   114  0.298246
1                   518  0.575290
**********************************************************************
AUROC comparison with imputation + missing-aware LLM variables
Structured only:                          0.709 ± 0.042
Structured + llm_present_3cat:           0.711 ± 0.041
Structured 

/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


# Stricter Check - New Gold Rule (Better Aligned with the Shock Prompt)

In [21]:
def normalize_chunks(chunks):
    if pd.isna(chunks):
        return ""
    if isinstance(chunks, str):
        try:
            parsed = ast.literal_eval(chunks)
            if isinstance(parsed, list):
                return " ".join(str(x) for x in parsed)
        except:
            pass
        return chunks
    if isinstance(chunks, list):
        return " ".join(str(x) for x in chunks)
    return str(chunks)


def normalize_llm_present(x):
    if pd.isna(x):
        return None
    if x is True or x is False:
        return x
    if isinstance(x, str):
        x = x.strip().lower()
        if x == "true":
            return True
        if x == "false":
            return False
    return None


def normalize_llm_severity(x):
    if pd.isna(x):
        return None
    if isinstance(x, str):
        x = x.strip().lower()
        if x in {"mild", "moderate", "severe"}:
            return x
    return None

def derive_shock_gold_label_prompt_aligned(text):
    t = str(text).lower().strip()
    if not t:
        return None

    if re.search(
        r"\b("
        r"no shock|not in shock|without shock|"
        r"shock resolved|resolved shock|"
        r"hemodynamically stable|"
        r"hemodynamically stable off pressors|"
        r"stable off pressors|"
        r"off vasopressors|off pressors|"
        r"no hypotension|not hypotensive|normotensive"
        r")\b",
        t
    ):
        return False

    if re.search(
        r"\b("
        r"cardiogenic shock|"
        r"septic shock|"
        r"distributive shock|"
        r"hemorrhagic shock|"
        r"vasodilatory shock|"
        r"hypovolemic shock|"
        r"obstructive shock|"
        r"shock state|"
        r"in shock|"
        r"remained in shock|"
        r"persistent shock|"
        r"refractory shock|"
        r"profound shock"
        r")\b",
        t
    ):
        return True

    if re.search(
        r"\b("
        r"circulatory failure|"
        r"hemodynamic instability|"
        r"hemodynamically unstable|"
        r"refractory hemodynamic instability"
        r")\b",
        t
    ):
        return True

    if re.search(
        r"\b("
        r"persistent hypotension|"
        r"continued hypotension|"
        r"remained hypotensive|"
        r"persistently hypotensive"
        r")\b",
        t
    ):
        return True

    if re.search(
        r"\b("
        r"required vasopressors|"
        r"requiring vasopressors|"
        r"on vasopressors|"
        r"vasopressor requirement|"
        r"vasopressor support|"
        r"vasopressor-dependent|"
        r"pressor-dependent|"
        r"required pressors|"
        r"needed pressors|"
        r"on pressors|"
        r"pressor support|"
        r"levophed|"
        r"norepinephrine"
        r")\b",
        t
    ):
        return True

    weak_agent = re.search(
        r"\b(dopamine|dobutamine|phenylephrine|epinephrine infusion)\b",
        t
    )
    instability_context = re.search(
        r"\b("
        r"shock|"
        r"hypotension|hypotensive|"
        r"hemodynamic instability|hemodynamically unstable|"
        r"circulatory failure|"
        r"poor perfusion"
        r")\b",
        t
    )
    if weak_agent and instability_context:
        return True

    if re.search(r"\b(hypotension|hypotensive)\b", t):
        return None

    if re.search(r"\bshock\b", t):
        return True

    return None


def derive_shock_gold_severity_prompt_aligned(text):
    t = str(text).lower().strip()
    if not t:
        return None

    if re.search(
        r"\b("
        r"no shock|not in shock|without shock|"
        r"shock resolved|resolved shock|"
        r"hemodynamically stable|"
        r"stable off pressors|off vasopressors|off pressors|"
        r"no hypotension|not hypotensive|normotensive"
        r")\b",
        t
    ):
        return "none"

    if re.search(
        r"\b("
        r"refractory shock|"
        r"profound shock|"
        r"cardiogenic shock|"
        r"septic shock|"
        r"multiple vasopressors|"
        r"max dose pressors|"
        r"high dose pressors|"
        r"vasopressor-dependent|"
        r"pressor-dependent|"
        r"refractory hemodynamic instability|"
        r"circulatory failure"
        r")\b",
        t
    ):
        return "severe"

    if re.search(
        r"\b("
        r"persistent hypotension|"
        r"continued hypotension|"
        r"remained hypotensive|"
        r"persistently hypotensive|"
        r"required vasopressors|"
        r"requiring vasopressors|"
        r"on vasopressors|"
        r"vasopressor support|"
        r"required pressors|"
        r"needed pressors|"
        r"on pressors|"
        r"pressor support|"
        r"hemodynamic instability|"
        r"hemodynamically unstable|"
        r"levophed|"
        r"norepinephrine"
        r")\b",
        t
    ):
        return "moderate"

    if re.search(
        r"\b("
        r"transient hypotension|"
        r"brief hypotension|"
        r"soft blood pressures|"
        r"low blood pressure"
        r")\b",
        t
    ):
        return "mild"

    weak_agent = re.search(
        r"\b(dopamine|dobutamine|phenylephrine|epinephrine infusion)\b",
        t
    )
    instability_context = re.search(
        r"\b(hypotension|hypotensive|shock|hemodynamic instability|circulatory failure)\b",
        t
    )
    if weak_agent and instability_context:
        return "moderate"

    if re.search(r"\bshock\b", t):
        return "moderate"

    if re.search(r"\b(hypotension|hypotensive)\b", t):
        return None

    return None


df["chunk_text"] = df["chunks"].apply(normalize_chunks)

df["gold_present_prompt_aligned"] = df["chunk_text"].apply(derive_shock_gold_label_prompt_aligned)
df["gold_severity_prompt_aligned"] = df["chunk_text"].apply(derive_shock_gold_severity_prompt_aligned)

df["llm_present_norm"] = df["llm_present"].apply(normalize_llm_present)
df["llm_severity_norm"] = df["llm_severity"].apply(normalize_llm_severity)

# Mention detection
df["gold_mentioned_prompt_aligned"] = df["chunk_text"].str.lower().str.contains(
    r"\b(?:shock|hypotension|hypotensive|circulatory failure|hemodynamic instability|"
    r"hemodynamically unstable|vasopressors|pressors|levophed|norepinephrine|"
    r"dopamine|dobutamine|phenylephrine|epinephrine infusion)\b",
    regex=True,
    na=False
)
df["llm_mentioned"] = df["llm_present_norm"].notna()

print("=== Mention detection (prompt-aligned) ===")
print(classification_report(df["gold_mentioned_prompt_aligned"], df["llm_mentioned"], zero_division=0))
print(confusion_matrix(df["gold_mentioned_prompt_aligned"], df["llm_mentioned"]))

# Binary presence evaluation
compare_df = df[
    df["gold_present_prompt_aligned"].notna() &
    df["llm_present_norm"].notna()
].copy()

y_true = compare_df["gold_present_prompt_aligned"].astype(bool)
y_pred = compare_df["llm_present_norm"].astype(bool)

print("=== Shock / hypotension detection (prompt-aligned) ===")
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

binary_mismatches = compare_df[
    compare_df["gold_present_prompt_aligned"] != compare_df["llm_present_norm"]
][["note_id", "chunk_text", "gold_present_prompt_aligned", "llm_present_norm", "llm_evidence_quote"]]

# Severity evaluation
severity_compare_df = df[
    df["gold_severity_prompt_aligned"].notna() &
    df["llm_severity_norm"].notna()
].copy()

print("=== Shock severity evaluation (prompt-aligned) ===")
print(
    classification_report(
        severity_compare_df["gold_severity_prompt_aligned"],
        severity_compare_df["llm_severity_norm"],
        zero_division=0
    )
)
print(
    confusion_matrix(
        severity_compare_df["gold_severity_prompt_aligned"],
        severity_compare_df["llm_severity_norm"],
        labels=["none", "mild", "moderate", "severe"]
    )
)

severity_mismatches = severity_compare_df[
    severity_compare_df["gold_severity_prompt_aligned"] != severity_compare_df["llm_severity_norm"]
][["note_id", "chunk_text", "gold_severity_prompt_aligned", "llm_severity_norm", "llm_evidence_quote"]]

#display(binary_mismatches.head(20))
#display(severity_mismatches.head(20))

=== Mention detection (prompt-aligned) ===
              precision    recall  f1-score   support

       False       0.89      0.95      0.92       926
        True       0.92      0.84      0.88       692

    accuracy                           0.90      1618
   macro avg       0.90      0.89      0.90      1618
weighted avg       0.90      0.90      0.90      1618

[[876  50]
 [110 582]]
=== Shock / hypotension detection (prompt-aligned) ===
              precision    recall  f1-score   support

       False       0.20      0.35      0.26        49
        True       0.93      0.87      0.90       515

    accuracy                           0.83       564
   macro avg       0.57      0.61      0.58       564
weighted avg       0.87      0.83      0.85       564

[[ 17  32]
 [ 66 449]]
=== Shock severity evaluation (prompt-aligned) ===
              precision    recall  f1-score   support

        mild       0.00      0.00      0.00         0
    moderate       0.57      0.05      0.0

In [22]:
#for idx, row in binary_mismatches.head(30).iterrows():
#    print(row['chunk_text'])
#    print('gold: ', row['gold_present_prompt_aligned'])
#    print('llm_present: ', row['llm_present_norm'])
#    print('llm_evidence_quote: ', row['llm_evidence_quote'])
#    print('*******************************************************')

In [23]:
import re
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix


# -----------------------------
# Helper regex groups
# -----------------------------
ELECTRICAL_SHOCK_PATTERNS = (
    r"aed shock|aed advised shock|shock advised|shock delivered|"
    r"electric shock|electrical shock|defibrillat(?:ed|ion)|cardiovert(?:ed|ed)|"
    r"icd shock|icd shocks|device shock|shocked once|shocked x ?\d+|"
    r"received \d+ shock|received \d+ shocks|"
    r"shock therapy|external shocks|internal shocks|"
    r"dc shock|defibrillation shock|"
    r"shock with rosc|shocks to restore your heartbeat|"
    r"the icd will shock you|if your heart should stop again"
)

TRUE_HEMODYNAMIC_SHOCK_PATTERNS = (
    r"cardiogenic shock|"
    r"septic shock|"
    r"distributive shock|"
    r"hemorrhagic shock|"
    r"vasodilatory shock|"
    r"hypovolemic shock|"
    r"obstructive shock|"
    r"mixed shock|"
    r"anaphylactic shock|"
    r"shock state|"
    r"in shock|"
    r"remained in shock|"
    r"persistent shock|"
    r"refractory shock|"
    r"profound shock|"
    r"undi[ -]?fferentiated shock"
)

INSTABILITY_PATTERNS = (
    r"circulatory failure|"
    r"hemodynamic instability|"
    r"hemodynamically unstable|"
    r"refractory hemodynamic instability|"
    r"poor perfusion"
)

HYPOTENSION_PATTERNS = (
    r"persistent hypotension|"
    r"continued hypotension|"
    r"remained hypotensive|"
    r"persistently hypotensive"
)

PRESSOR_PATTERNS = (
    r"required vasopressors|"
    r"requiring vasopressors|"
    r"on vasopressors|"
    r"vasopressor requirement|"
    r"vasopressor support|"
    r"vasopressor-dependent|"
    r"pressor-dependent|"
    r"required pressors|"
    r"needed pressors|"
    r"on pressors|"
    r"pressor support|"
    r"multiple vasopressors|"
    r"max dose pressors|"
    r"high dose pressors|"
    r"levophed|"
    r"norepinephrine|"
    r"vasopressin"
)

WEAK_AGENT_PATTERNS = (
    r"dopamine|dobutamine|phenylephrine|epinephrine infusion"
)

NEGATION_OR_RESOLUTION_PATTERNS = (
    r"no shock|not in shock|without shock|"
    r"shock resolved|resolved shock|"
    r"hemodynamically stable|"
    r"hemodynamically stable off pressors|"
    r"stable off pressors|"
    r"off vasopressors|off pressors|"
    r"without pressor requirement|no pressor requirement|"
    r"no hypotension|not hypotensive|normotensive|"
    r"remained stable|clinically improved|stable condition"
)

MILD_PATTERNS = (
    r"transient hypotension|"
    r"brief hypotension|"
    r"soft blood pressures|"
    r"low blood pressure"
)


def has(pattern: str, text: str) -> bool:
    return bool(re.search(rf"\b(?:{pattern})\b", text))


def electrical_shock_only_context(text: str) -> bool:
    """
    True when there is electrical shock language but no convincing evidence
    of hemodynamic / circulatory shock.
    """
    has_electrical = has(ELECTRICAL_SHOCK_PATTERNS, text)
    has_hemodynamic = (
        has(TRUE_HEMODYNAMIC_SHOCK_PATTERNS, text)
        or has(INSTABILITY_PATTERNS, text)
        or has(HYPOTENSION_PATTERNS, text)
        or has(PRESSOR_PATTERNS, text)
    )
    return has_electrical and not has_hemodynamic


def derive_shock_gold_label_prompt_aligned_refined(text):
    t = str(text).lower().strip()
    if not t:
        return None

    # 1) Pure electrical shock language should not count as circulatory shock
    if electrical_shock_only_context(t):
        return False

    # 2) Strong positive hemodynamic evidence takes priority over later recovery wording
    strong_positive = (
        has(TRUE_HEMODYNAMIC_SHOCK_PATTERNS, t)
        or has(INSTABILITY_PATTERNS, t)
        or has(HYPOTENSION_PATTERNS, t)
        or has(PRESSOR_PATTERNS, t)
    )

    weak_agent = has(WEAK_AGENT_PATTERNS, t)
    instability_context = has(
        r"shock|hypotension|hypotensive|hemodynamic instability|"
        r"hemodynamically unstable|circulatory failure|poor perfusion",
        t
    )

    if weak_agent and instability_context:
        strong_positive = True

    if strong_positive:
        return True

    # 3) Only after checking for true positive evidence, allow explicit negation/resolution
    if has(NEGATION_OR_RESOLUTION_PATTERNS, t):
        return False

    # 4) Generic hypotension alone is ambiguous
    if has(r"hypotension|hypotensive", t):
        return None

    # 5) Generic 'shock' alone is also ambiguous unless it looks hemodynamic
    #    This avoids counting every 'shock' mention as positive.
    if has(r"shock", t):
        # if it is not electrical-only and not clearly negated/resolved,
        # treat generic shock as ambiguous rather than auto-True
        return None

    return None


def derive_shock_gold_severity_prompt_aligned_refined(text):
    t = str(text).lower().strip()
    if not t:
        return None

    # 1) Pure electrical shock language is not hemodynamic severity
    if electrical_shock_only_context(t):
        return "none"

    # 2) Strong severe patterns
    if has(
        r"refractory shock|"
        r"profound shock|"
        r"cardiogenic shock|"
        r"septic shock|"
        r"mixed shock|"
        r"anaphylactic shock|"
        r"multiple vasopressors|"
        r"max dose pressors|"
        r"high dose pressors|"
        r"vasopressor-dependent|"
        r"pressor-dependent|"
        r"refractory hemodynamic instability|"
        r"circulatory failure|"
        r"undi[ -]?fferentiated shock",
        t
    ):
        return "severe"

    # 3) Moderate patterns
    if has(
        r"persistent hypotension|"
        r"continued hypotension|"
        r"remained hypotensive|"
        r"persistently hypotensive|"
        r"required vasopressors|"
        r"requiring vasopressors|"
        r"on vasopressors|"
        r"vasopressor support|"
        r"required pressors|"
        r"needed pressors|"
        r"on pressors|"
        r"pressor support|"
        r"hemodynamic instability|"
        r"hemodynamically unstable|"
        r"levophed|"
        r"norepinephrine|"
        r"vasopressin",
        t
    ):
        return "moderate"

    # 4) Mild patterns
    if has(MILD_PATTERNS, t):
        return "mild"

    # 5) Weak agent + instability context
    weak_agent = has(WEAK_AGENT_PATTERNS, t)
    instability_context = has(
        r"hypotension|hypotensive|shock|hemodynamic instability|circulatory failure",
        t
    )
    if weak_agent and instability_context:
        return "moderate"

    # 6) Explicit negation / recovery only after positive evidence checks
    if has(NEGATION_OR_RESOLUTION_PATTERNS, t):
        return "none"

    # 7) Generic shock alone is ambiguous; don't force moderate
    if has(r"shock", t):
        return None

    # 8) Generic hypotension alone remains ambiguous
    if has(r"hypotension|hypotensive", t):
        return None

    return None


# -----------------------------------
# Apply refined gold labels
# -----------------------------------
df["chunk_text"] = df["chunks"].apply(normalize_chunks)

df["gold_present_prompt_aligned_refined"] = df["chunk_text"].apply(
    derive_shock_gold_label_prompt_aligned_refined
)
df["gold_severity_prompt_aligned_refined"] = df["chunk_text"].apply(
    derive_shock_gold_severity_prompt_aligned_refined
)

df["llm_present_norm"] = df["llm_present"].apply(normalize_llm_present)
df["llm_severity_norm"] = df["llm_severity"].apply(normalize_llm_severity)

# Mention detection: focus on hemodynamic shock concepts, not electrical shocks
df["gold_mentioned_prompt_aligned_refined"] = df["chunk_text"].str.lower().str.contains(
    r"\b(?:"
    r"cardiogenic shock|septic shock|distributive shock|hemorrhagic shock|"
    r"vasodilatory shock|hypovolemic shock|obstructive shock|mixed shock|"
    r"anaphylactic shock|shock state|circulatory failure|hemodynamic instability|"
    r"hemodynamically unstable|persistent hypotension|continued hypotension|"
    r"remained hypotensive|persistently hypotensive|vasopressors|pressors|"
    r"levophed|norepinephrine|vasopressin|dopamine|dobutamine|phenylephrine|"
    r"epinephrine infusion"
    r")\b",
    regex=True,
    na=False
)

df["llm_mentioned"] = df["llm_present_norm"].notna()

print("=== Mention detection (prompt-aligned refined) ===")
print(classification_report(df["gold_mentioned_prompt_aligned_refined"], df["llm_mentioned"], zero_division=0))
print(confusion_matrix(df["gold_mentioned_prompt_aligned_refined"], df["llm_mentioned"]))

# Binary presence evaluation
compare_df = df[
    df["gold_present_prompt_aligned_refined"].notna() &
    df["llm_present_norm"].notna()
].copy()

y_true = compare_df["gold_present_prompt_aligned_refined"].astype(bool)
y_pred = compare_df["llm_present_norm"].astype(bool)

print("=== Shock / hypotension detection (prompt-aligned refined) ===")
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

binary_mismatches = compare_df[
    compare_df["gold_present_prompt_aligned_refined"] != compare_df["llm_present_norm"]
][[
    "note_id", "chunk_text",
    "gold_present_prompt_aligned_refined",
    "llm_present_norm",
    "llm_evidence_quote"
]]

# Severity evaluation
severity_compare_df = df[
    df["gold_severity_prompt_aligned_refined"].notna() &
    df["llm_severity_norm"].notna()
].copy()

print("=== Shock severity evaluation (prompt-aligned refined) ===")
print(
    classification_report(
        severity_compare_df["gold_severity_prompt_aligned_refined"],
        severity_compare_df["llm_severity_norm"],
        zero_division=0
    )
)
print(
    confusion_matrix(
        severity_compare_df["gold_severity_prompt_aligned_refined"],
        severity_compare_df["llm_severity_norm"],
        labels=["none", "mild", "moderate", "severe"]
    )
)

severity_mismatches = severity_compare_df[
    severity_compare_df["gold_severity_prompt_aligned_refined"] != severity_compare_df["llm_severity_norm"]
][[
    "note_id", "chunk_text",
    "gold_severity_prompt_aligned_refined",
    "llm_severity_norm",
    "llm_evidence_quote"
]]

#display(binary_mismatches.head(20))
# display(severity_mismatches.head(20))

=== Mention detection (prompt-aligned refined) ===
              precision    recall  f1-score   support

       False       0.97      0.84      0.90      1144
        True       0.71      0.94      0.81       474

    accuracy                           0.87      1618
   macro avg       0.84      0.89      0.85      1618
weighted avg       0.89      0.87      0.87      1618

[[959 185]
 [ 27 447]]
=== Shock / hypotension detection (prompt-aligned refined) ===
              precision    recall  f1-score   support

       False       0.61      0.56      0.58        68
        True       0.93      0.94      0.94       434

    accuracy                           0.89       502
   macro avg       0.77      0.75      0.76       502
weighted avg       0.89      0.89      0.89       502

[[ 38  30]
 [ 24 410]]
=== Shock severity evaluation (prompt-aligned refined) ===
              precision    recall  f1-score   support

    moderate       0.58      0.10      0.17        68
        none      

In [24]:
#for idx, row in binary_mismatches.head(30).iterrows():
#    print(row['chunk_text'])
#    print('gold: ', row['gold_present_prompt_aligned_refined'])
#    print('llm_present: ', row['llm_present_norm'])
#    print('llm_evidence_quote: ', row['llm_evidence_quote'])
#    print('*******************************************************')

In [25]:
import re
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix


def has_any(text: str, patterns) -> bool:
    return any(re.search(p, text) for p in patterns)


# -----------------------------
# Pattern groups
# -----------------------------

ELECTRICAL_SHOCK_PATTERNS = [
    r"\baed advised shock\b",
    r"\bshock advised\b",
    r"\bshock delivered\b",
    r"\belectric(?:al)? shock\b",
    r"\bshock therapy\b",
    r"\bdefibrillat(?:ed|ion)\b",
    r"\bcardiovert(?:ed|ersion)\b",
    r"\bdc shock\b",
    r"\bicd shock(?:s)?\b",
    r"\bdevice shock(?:s)?\b",
    r"\bexternal shock(?:s)?\b",
    r"\binternal shock(?:s)?\b",
    r"\bshocked x ?\d+\b",
    r"\bshocked once\b",
    r"\bshock with rosc\b",
    r"\bshock(?:s)? to restore (?:your )?heartbeat\b",
    r"\bthe icd will shock you\b",
    r"\bshockable rhythm\b",
]

DIRECT_HEMODYNAMIC_SHOCK_PATTERNS = [
    r"\bcardiogenic shock\b",
    r"\bseptic shock\b",
    r"\bdistributive shock\b",
    r"\bhemorrhagic shock\b",
    r"\bhypovolemic shock\b",
    r"\bobstructive shock\b",
    r"\bvasodilatory shock\b",
    r"\bmixed shock\b",
    r"\banaphylactic shock\b",
    r"\bundifferentiated shock\b",
    r"\bshock state\b",
    r"\bin shock\b",
    r"\bremained in shock\b",
    r"\bpersistent shock\b",
    r"\brefractory shock\b",
    r"\bprofound shock\b",
    r"\bcirculatory failure\b",
    r"\bhemodynamic instability\b",
    r"\bhemodynamically unstable\b",
    r"\brefractory hemodynamic instability\b",
]

PRESSOR_PATTERNS = [
    r"\brequired vasopressors\b",
    r"\brequiring vasopressors\b",
    r"\bon vasopressors\b",
    r"\bvasopressor requirement\b",
    r"\bvasopressor support\b",
    r"\bvasopressor-dependent\b",
    r"\bpressor-dependent\b",
    r"\brequired pressors\b",
    r"\bneeded pressors\b",
    r"\bon pressors\b",
    r"\bpressor support\b",
    r"\bmultiple vasopressors\b",
    r"\bmax dose pressors\b",
    r"\bhigh dose pressors\b",
    r"\blevophed\b",
    r"\bnorepinephrine\b",
    r"\bvasopressin\b",
]

WEAK_AGENT_PATTERNS = [
    r"\bdopamine\b",
    r"\bdobutamine\b",
    r"\bphenylephrine\b",
    r"\bepinephrine infusion\b",
]

HYPOTENSION_PATTERNS = [
    r"\bpersistent hypotension\b",
    r"\bcontinued hypotension\b",
    r"\bremained hypotensive\b",
    r"\bpersistently hypotensive\b",
]

GENERIC_HYPOTENSION_PATTERNS = [
    r"\bhypotension\b",
    r"\bhypotensive\b",
]

NEGATION_RESOLUTION_PATTERNS = [
    r"\bno shock\b",
    r"\bnot in shock\b",
    r"\bwithout shock\b",
    r"\bshock resolved\b",
    r"\bresolved shock\b",
    r"\bhemodynamically stable\b",
    r"\bhemodynamically stable off pressors\b",
    r"\bstable off pressors\b",
    r"\boff vasopressors\b",
    r"\boff pressors\b",
    r"\bwithout pressor requirement\b",
    r"\bno pressor requirement\b",
    r"\bno hypotension\b",
    r"\bnot hypotensive\b",
    r"\bnormotensive\b",
    r"\bremained stable\b",
    r"\bstable condition\b",
    r"\bclinically improved\b",
]

SHOCK_LIVER_ONLY_PATTERNS = [
    r"\bshock liver\b",
    r"\blikely shock liver\b",
    r"\bin the setting of shock\b",
    r"\bdue to shock\b",
    r"\bsecondary to shock\b",
    r"\btransaminitis.*shock\b",
    r"\bcoagulopathy.*shock\b",
]

MILD_PATTERNS = [
    r"\btransient hypotension\b",
    r"\bbrief hypotension\b",
    r"\bsoft blood pressures\b",
    r"\blow blood pressure\b",
]


# -----------------------------
# Helper detectors
# -----------------------------

def has_direct_hemodynamic_evidence(t: str) -> bool:
    if has_any(t, DIRECT_HEMODYNAMIC_SHOCK_PATTERNS):
        return True
    if has_any(t, PRESSOR_PATTERNS):
        return True
    if has_any(t, HYPOTENSION_PATTERNS):
        return True

    weak_agent = has_any(t, WEAK_AGENT_PATTERNS)
    instability_context = has_any(t, [
        r"\bshock\b",
        r"\bhypotension\b",
        r"\bhypotensive\b",
        r"\bhemodynamic instability\b",
        r"\bhemodynamically unstable\b",
        r"\bcirculatory failure\b",
        r"\bpoor perfusion\b",
    ])
    if weak_agent and instability_context:
        return True

    return False


def is_electrical_shock_only(t: str) -> bool:
    return has_any(t, ELECTRICAL_SHOCK_PATTERNS) and not has_direct_hemodynamic_evidence(t)


def is_shock_liver_only(t: str) -> bool:
    return has_any(t, SHOCK_LIVER_ONLY_PATTERNS) and not has_direct_hemodynamic_evidence(t)


# -----------------------------
# Binary gold
# -----------------------------

def derive_shock_gold_label_strict(text):
    t = str(text).lower().strip()
    if not t:
        return None

    # 1) Direct hemodynamic evidence always wins
    if has_direct_hemodynamic_evidence(t):
        return True

    # 2) Electrical shock only is not circulatory shock
    if is_electrical_shock_only(t):
        return None

    # 3) Shock-liver-only / downstream consequence is ambiguous
    if is_shock_liver_only(t):
        return None

    # 4) Explicit recovery / no-shock statements only count as False
    #    if there was no earlier positive evidence
    if has_any(t, NEGATION_RESOLUTION_PATTERNS):
        return False

    # 5) Generic shock alone is ambiguous
    if re.search(r"\bshock\b", t):
        return None

    # 6) Generic hypotension alone is ambiguous
    if has_any(t, GENERIC_HYPOTENSION_PATTERNS):
        return None

    return None


# -----------------------------
# Severity gold
# -----------------------------

def derive_shock_gold_severity_strict(text):
    t = str(text).lower().strip()
    if not t:
        return None

    # Direct severe evidence
    if has_any(t, [
        r"\brefractory shock\b",
        r"\bprofound shock\b",
        r"\bcardiogenic shock\b",
        r"\bseptic shock\b",
        r"\bmixed shock\b",
        r"\banaphylactic shock\b",
        r"\bmultiple vasopressors\b",
        r"\bmax dose pressors\b",
        r"\bhigh dose pressors\b",
        r"\bvasopressor-dependent\b",
        r"\bpressor-dependent\b",
        r"\brefractory hemodynamic instability\b",
        r"\bcirculatory failure\b",
        r"\bundifferentiated shock\b",
    ]):
        return "severe"

    # Moderate evidence
    if has_any(t, HYPOTENSION_PATTERNS) or has_any(t, PRESSOR_PATTERNS) or has_any(t, [
        r"\bhemodynamic instability\b",
        r"\bhemodynamically unstable\b",
    ]):
        return "moderate"

    weak_agent = has_any(t, WEAK_AGENT_PATTERNS)
    instability_context = has_any(t, [
        r"\bhypotension\b",
        r"\bhypotensive\b",
        r"\bshock\b",
        r"\bhemodynamic instability\b",
        r"\bcirculatory failure\b",
    ])
    if weak_agent and instability_context:
        return "moderate"

    # Mild evidence
    if has_any(t, MILD_PATTERNS):
        return "mild"

    # Electrical-shock-only or shock-liver-only should not be severity-labeled
    if is_electrical_shock_only(t):
        return None

    if is_shock_liver_only(t):
        return None

    # Explicit no-shock / resolved only if no positive evidence above
    if has_any(t, NEGATION_RESOLUTION_PATTERNS):
        return "none"

    # Generic shock or hypotension alone stays ambiguous
    if re.search(r"\bshock\b", t):
        return None

    if has_any(t, GENERIC_HYPOTENSION_PATTERNS):
        return None

    return None


# -----------------------------
# Apply
# -----------------------------

df["chunk_text"] = df["chunks"].apply(normalize_chunks)

df["gold_present_prompt_aligned_strict"] = df["chunk_text"].apply(derive_shock_gold_label_strict)
df["gold_severity_prompt_aligned_strict"] = df["chunk_text"].apply(derive_shock_gold_severity_strict)

df["llm_present_norm"] = df["llm_present"].apply(normalize_llm_present)
df["llm_severity_norm"] = df["llm_severity"].apply(normalize_llm_severity)

# Mention detection: only hemodynamic / circulatory failure concepts
df["gold_mentioned_prompt_aligned_strict"] = df["chunk_text"].str.lower().str.contains(
    r"\b(?:"
    r"cardiogenic shock|septic shock|distributive shock|hemorrhagic shock|"
    r"vasodilatory shock|hypovolemic shock|obstructive shock|mixed shock|"
    r"anaphylactic shock|undifferentiated shock|shock state|circulatory failure|"
    r"hemodynamic instability|hemodynamically unstable|persistent hypotension|"
    r"continued hypotension|remained hypotensive|persistently hypotensive|"
    r"vasopressors|pressors|levophed|norepinephrine|vasopressin|"
    r"dopamine|dobutamine|phenylephrine|epinephrine infusion"
    r")\b",
    regex=True,
    na=False
)

df["llm_mentioned"] = df["llm_present_norm"].notna()

print("=== Mention detection (strict) ===")
print(classification_report(df["gold_mentioned_prompt_aligned_strict"], df["llm_mentioned"], zero_division=0))
print(confusion_matrix(df["gold_mentioned_prompt_aligned_strict"], df["llm_mentioned"]))

# Binary evaluation
compare_df = df[
    df["gold_present_prompt_aligned_strict"].notna() &
    df["llm_present_norm"].notna()
].copy()

y_true = compare_df["gold_present_prompt_aligned_strict"].astype(bool)
y_pred = compare_df["llm_present_norm"].astype(bool)

print("=== Shock / hypotension detection (strict) ===")
print(classification_report(y_true, y_pred, zero_division=0))
print(confusion_matrix(y_true, y_pred))

binary_mismatches = compare_df[
    compare_df["gold_present_prompt_aligned_strict"] != compare_df["llm_present_norm"]
][[
    "note_id", "chunk_text",
    "gold_present_prompt_aligned_strict",
    "llm_present_norm",
    "llm_evidence_quote"
]]

# Severity evaluation
severity_compare_df = df[
    df["gold_severity_prompt_aligned_strict"].notna() &
    df["llm_severity_norm"].notna()
].copy()

print("=== Shock severity evaluation (strict) ===")
print(
    classification_report(
        severity_compare_df["gold_severity_prompt_aligned_strict"],
        severity_compare_df["llm_severity_norm"],
        zero_division=0
    )
)
print(
    confusion_matrix(
        severity_compare_df["gold_severity_prompt_aligned_strict"],
        severity_compare_df["llm_severity_norm"],
        labels=["none", "mild", "moderate", "severe"]
    )
)

severity_mismatches = severity_compare_df[
    severity_compare_df["gold_severity_prompt_aligned_strict"] != severity_compare_df["llm_severity_norm"]
][[
    "note_id", "chunk_text",
    "gold_severity_prompt_aligned_strict",
    "llm_severity_norm",
    "llm_evidence_quote"
]]

#display(binary_mismatches.head(20))
# display(severity_mismatches.head(20))

=== Mention detection (strict) ===
              precision    recall  f1-score   support

       False       0.97      0.84      0.90      1140
        True       0.71      0.94      0.81       478

    accuracy                           0.87      1618
   macro avg       0.84      0.89      0.86      1618
weighted avg       0.89      0.87      0.87      1618

[[958 182]
 [ 28 450]]
=== Shock / hypotension detection (strict) ===
              precision    recall  f1-score   support

       False       0.26      0.73      0.38        11
        True       0.99      0.95      0.97       433

    accuracy                           0.94       444
   macro avg       0.63      0.84      0.68       444
weighted avg       0.97      0.94      0.95       444

[[  8   3]
 [ 23 410]]
=== Shock severity evaluation (strict) ===
              precision    recall  f1-score   support

    moderate       0.58      0.10      0.17        69
        none       0.00      0.00      0.00         8
      severe